# WoodelfHD — Depth Sweep

Runs **WoodelfHD** (`woodelf_for_high_depth`) across all datasets and task types
in the `woodelfhd_depth_sweep_experiment`.  
This notebook is **independent** and can run in parallel with the other method notebooks.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Clones `treebranchmarks` repo
3. Installs all dependencies
4. Runs `woodelfhd_depth_sweep_experiment --method woodelf_hd`
5. Writes partial results to Drive as `woodelf_hd.json`

### Datasets (all download automatically)
| Dataset | Source |
|---------|--------|
| Fraud Detection | Google Drive parquet (~200 MB) |
| HIGGS | Google Drive parquet |
| KDD Cup (Intrusion Detection) | Google Drive parquet |
| California Housing | sklearn builtin |

> **Runtime estimate:** Expect several hours on a standard Colab CPU runtime.

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
# Edit DRIVE_FOLDER to point to your preferred Google Drive folder.
# All other paths are derived from it.
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'woodelf_hd.json'
print(f'Results will be saved to: {DRIVE_RESULT_PATH}')

Results will be saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/woodelf_hd.json


In [ ]:
# ── Step 3: Clone repository ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks

Cloning into '/content/treebranchmarks'...
remote: Enumerating objects: 367, done.
remote: Counting objects: 100% (367/367), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 367 (delta 233), reused 270 (delta 139), pack-reused 0 (from 0)
Receiving objects: 100% (367/367), 220.59 KiB | 16.97 MiB/s, done.
Resolving deltas: 100% (233/233), done.


In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
# woodelf_explainer must be installed before treebranchmarks (it is listed
# as a dependency in treebranchmarks/pyproject.toml).

!pip install woodelf_explainer

!pip install -q -e /content/treebranchmarks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.0 MB/s eta 0:00:00
  Building editable for treebranchmarks (pyproject.toml) ... done


In [ ]:
# ── Step 5: Restore method cache from a previous interrupted run ─────────────
# The framework writes the method cache file to Drive after every approach result.
# On restart, copy it back to the local cache directory so the experiment
# skips already-completed entries and only runs what is still missing.
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)
local_cache_file = cache_dir / 'woodelf_hd.json'

if DRIVE_RESULT_PATH.exists() and not local_cache_file.exists():
    shutil.copy(DRIVE_RESULT_PATH, local_cache_file)
    print(f'Restored method cache ({DRIVE_RESULT_PATH.stat().st_size // 1024} KB)')
else:
    print('No method cache to restore — starting fresh.')

Restored method cache (29 KB)


In [ ]:
# ── Step 6: Run the experiment (WoodelfHD only) ───────────────────────────────
# -u   : unbuffered output so progress prints appear in real time
# --method woodelf_hd : only the WoodelfHD approach is timed
# --result_location   : dual-write to Drive after every completed mission

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method woodelf_hd \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.
Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100% 69.6M/69.6M [00:00<00:00, 126MB/s]
[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 15.44s — T=100, D=6, L=32.0, F=397
  [approach:WoodelfHD] CACHED=3.409s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 18.19s — T=100, D=9, L=82.8, F=397
  [approach:WoodelfHD] CACHED=12.948s

  > D=12  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 24.91s — T=100, D=12, L=154.0, F=397
  [approach:WoodelfHD] CACHED=35.725s

  > D=15  n=118108 

In [ ]:
# ── Step 7: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    cache = json.load(f)

print(f'Entries in method cache: {len(cache)}')
if cache:
    sample = next(iter(cache.values()))
    print(f'Sample entry: {sample["_label"]}  →  {sample["running_time"]:.3f}s')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')

Entries in method cache: 94
Sample entry: Path-Dependent SHAP n=118108 m=0 D=6 T=100  →  3.409s

File saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/woodelf_hd.json


# Old Runs

In [ ]:
# ── Step 6: Run the experiment (WoodelfHD only) ───────────────────────────────
# -u   : unbuffered output so progress prints appear in real time
# --method woodelf_hd : only the WoodelfHD approach is timed
# --result_location   : dual-write to Drive after every completed mission

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method woodelf_hd \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.
Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100% 69.6M/69.6M [00:00<00:00, 175MB/s]
[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 15.02s — T=100, D=6, L=32.0, F=397
  [approach:WoodelfHD] CACHED=3.409s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 17.48s — T=100, D=9, L=82.8, F=397
  [approach:WoodelfHD] CACHED=12.948s

  > D=12  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 21.57s — T=100, D=12, L=154.0, F=397
  [approach:WoodelfHD] CACHED=35.725s

  > D=15  n=118108 

In [ ]:
# ── Step 6: Run the experiment (WoodelfHD only) ───────────────────────────────
# -u   : unbuffered output so progress prints appear in real time
# --method woodelf_hd : only the WoodelfHD approach is timed
# --result_location   : dual-write to Drive after every completed mission

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method woodelf_hd \
    --result_location "{DRIVE_RESULT_PATH}"

/content/treebranchmarks

Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.
Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100% 69.6M/69.6M [00:00<00:00, 151MB/s]
[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 16.35s — T=100, D=6, L=32.0, F=397
  [approach:WoodelfHD] CACHED=3.409s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 19.48s — T=100, D=9, L=82.8, F=397
  [approach:WoodelfHD] CACHED=12.948s

  > D=12  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 23.39s — T=100, D=12, L=154.0, F=397
  [approach:WoodelfHD] CACHED=35.725s

  > D=15  n=118108 